# Group Relative Policy Optimization (GRPO) with LoRA using TRL — on Intel XPU

Adapted from `grpo_trl_lora_qlora.ipynb` for Intel XPU (torch-only, no IPEX).

![trl banner](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)

Fine-tune **Large Language Models (LLMs)** with **LoRA** using [**TRL**](https://github.com/huggingface/trl) and GRPO on **Intel XPU** devices.

### Key differences from the CUDA version:
- **No `bitsandbytes`** — not supported on XPU. We use LoRA (not QLoRA) with `bfloat16`.
- **No `liger-kernel`** — CUDA-only. Removed.
- **No `paged_adamw_8bit`** — requires bitsandbytes. We use `adamw_torch` instead.
- All `torch.cuda.*` calls replaced with `torch.xpu.*`.
- `device_map="auto"` replaced with explicit XPU placement.
- Uses upstream PyTorch with native `torch.xpu` support (2.4+). No IPEX needed.

## Verify XPU availability

In [ ]:
import torch

assert torch.xpu.is_available(), "XPU is not available! Check your PyTorch installation."
print(f"PyTorch version: {torch.__version__}")
print(f"XPU device count: {torch.xpu.device_count()}")
for i in range(torch.xpu.device_count()):
    print(f"  Device {i}: {torch.xpu.get_device_name(i)}")

## Install dependencies

Install **TRL** with the **PEFT** extra. No `bitsandbytes` or `liger-kernel` needed for XPU.

In [ ]:
!pip install -Uq "trl[peft]" math_verify

### Log in to Hugging Face

Log in to your **Hugging Face** account to save your fine-tuned model or access gated models. You can find your **access token** on your [account settings page](https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Load Dataset

We load the [**AI-MO/NuminaMath-TIR**](https://huggingface.co/datasets/AI-MO/NuminaMath-TIR) dataset — mathematical reasoning problems requiring step-by-step solutions.

For efficiency, we load only a small portion of the training split:

In [ ]:
from datasets import load_dataset

dataset_name = 'AI-MO/NuminaMath-TIR'
train_dataset = load_dataset(dataset_name, split='train[:5%]')

In [ ]:
print(train_dataset)

In [ ]:
print(train_dataset[0])

Adapt the dataset to a conversational format with a system prompt guiding the model to produce step-by-step reasoning:

In [ ]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant  "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process is enclosed strictly within <think> and </think> tags. "
    "After closing </think>, the assistant MUST provide the final answer in plain text."
)


def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

train_dataset = train_dataset.map(make_conversation)

In [ ]:
print(train_dataset[0]['prompt'])

In [ ]:
train_dataset = train_dataset.remove_columns(['messages', 'problem'])
print(train_dataset)

## Select model

Choose your preferred model below. Since we use LoRA (no quantization) with bfloat16, memory usage will be higher than the QLoRA version.

In [ ]:
# Select one model below by uncommenting the line you want to use
## Qwen
model_id, output_dir = "Qwen/Qwen2-7B-Instruct", "xpu-Qwen2-7B-Instruct-GRPO"
# model_id, output_dir = "Qwen/Qwen3-8B", "xpu-Qwen3-8B-GRPO"
# model_id, output_dir = "Qwen/Qwen2.5-7B-Instruct", "xpu-Qwen2.5-7B-Instruct-GRPO"

## Llama
# model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "xpu-Llama-3.2-3B-Instruct-GRPO"
# model_id, output_dir = "meta-llama/Llama-3.1-8B-Instruct", "xpu-Llama-3.1-8B-Instruct-GRPO"

## Load model for XPU

We load the model in **bfloat16** without quantization. On XPU we avoid `device_map="auto"` and place the model explicitly.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",
    torch_dtype=torch.bfloat16,
).to("xpu")

## Configure LoRA

LoRA works on any device — we fine-tune a lightweight adapter instead of modifying the full model weights.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## Train model

We use `think_format_reward` and `reasoning_accuracy_reward` from `trl.rewards`.

In [ ]:
from trl.rewards import think_format_reward, reasoning_accuracy_reward

Configure GRPO training. Key XPU differences:
- `optim="adamw_torch"` instead of `paged_adamw_8bit` (no bitsandbytes)
- `use_liger_kernel=False` (CUDA-only)
- `bf16=True` for bfloat16 training on XPU

In [ ]:
from trl import GRPOConfig

training_args = GRPOConfig(
    # Training schedule / optimization
    learning_rate=2e-5,
    max_steps=500,

    # GRPO parameters
    per_device_train_batch_size=8,
    max_completion_length=256,
    num_generations=8,

    # Optimizations for XPU
    optim="adamw_torch",                # paged_adamw_8bit requires bitsandbytes (CUDA-only)
    bf16=True,                           # Use bfloat16 on XPU
    gradient_checkpointing=True,         # Reduce memory usage

    # Reporting and saving
    output_dir=output_dir,
    logging_steps=10,
    report_to="none",
    log_completions=False,

    # Hub integration
    push_to_hub=False,
)

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[think_format_reward, reasoning_accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

### Show XPU memory stats before training

In [ ]:
xpu_stats = torch.xpu.get_device_properties(0)
start_xpu_memory = round(torch.xpu.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(xpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"XPU = {xpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_xpu_memory} GB of memory reserved.")

### Train!

In [ ]:
trainer_stats = trainer.train()

### Show XPU memory stats after training

In [ ]:
used_memory = round(torch.xpu.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_xpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Save fine-tuned model

In [ ]:
trainer.save_model(output_dir)

## Load the fine-tuned model and run inference

Load the base model on XPU, then attach the LoRA adapter.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_model = output_dir

base_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16
).to("xpu")

tokenizer = AutoTokenizer.from_pretrained(model_id)

Load a test sample:

In [ ]:
from datasets import load_dataset

dataset_name = 'AI-MO/NuminaMath-TIR'
test_dataset = load_dataset(dataset_name, split='test[:1%]')
test_dataset = test_dataset.map(make_conversation)
test_dataset = test_dataset.remove_columns(['messages', 'problem'])
test_dataset[0]['prompt']

Test the **base model** (without adapter):

In [ ]:
messages = test_dataset[0]['prompt']
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(
    **model_inputs,
    max_new_tokens=256
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

Test the **fine-tuned model** (with LoRA adapter):

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_model)

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=256
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

## Merge and save (optional)

Merge the LoRA adapter into the base model for standalone deployment:

In [ ]:
model_merged = fine_tuned_model.merge_and_unload()

save_dir = f"{output_dir}-merged"

model_merged.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)